# Phase I і Phase II на виправлених сплітах (для ревізії статті)

**Запуск:** Run All. Наприкінці буде один підсумковий звіт, його й треба скопіювати. Тривалість приблизно 5–15 хв.

- **UNSW-NB15** — хронологічний спліт статті. **CIC-IDS2017** — виправлені таймстемпи та стратифікований часовий спліт.
- **Phase I** рахується лише на train-частині: H(C), H(C|F), MI, NMI і ρ для трьох пар:
  - cve_ctx_service проти нативного поля (service/protocol);
  - cve_ctx_service проти (протокол, порт призначення);
  - усі CTI-поля разом проти (протокол, порт призначення).
- **Phase II** — частка gain для моделей F0 і F1 при 500 деревах, тобто за тим самим протоколом, що й в експерименті B.

Контроль: AP моделей F0 і F1 має збігтися з експериментом B (UNSW 0.982439 / 0.983854; CIC 0.998869 / 0.999402).

In [1]:
import os, gc, re, time, warnings
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
warnings.filterwarnings("ignore")

# ---------------- конфігурація ----------------
OUT = "/srv/ids_research/data/phase12"
CACHE = os.path.join(OUT, "cache")
os.makedirs(CACHE, exist_ok=True)
UNSW_PATH = "/srv/ids_research/data/enriched/unsw_nb15_with_cve_context_dedup.parquet"
CIC_PATH = "/srv/ids_research/data/enriched/cic_ids2017_with_cve_context.parquet"
SEED, ROUNDS, N_BOOT = 42, 500, 500
FPR_TARGET, TOP_PORTS, MIN_SVC_BENIGN, Z_CLIP, CHUNK = 0.001, 200, 50, 50.0, 400_000

F0_UNSW = ['state','dur','sbytes','dbytes','sttl','dttl','sloss','dloss','Sload','Dload','Spkts','Dpkts',
'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth','res_bdy_len','Sjit','Djit','Sintpkt',
'Dintpkt','tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl','ct_flw_http_mthd','is_ftp_login',
'ct_ftp_cmd','ct_srv_src','ct_srv_dst','ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
'ct_dst_src_ltm','proto','service']
CTX_UNSW = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_epss_coverage',
            'cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']
F0_CIC = ['Flow Duration','Total Fwd Packets','Total Backward Packets','Total Length of Fwd Packets',
'Total Length of Bwd Packets','Fwd Packet Length Max','Fwd Packet Length Min','Fwd Packet Length Mean',
'Fwd Packet Length Std','Bwd Packet Length Max','Bwd Packet Length Min','Bwd Packet Length Mean',
'Bwd Packet Length Std','Flow Bytes/s','Flow Packets/s','Flow IAT Mean','Flow IAT Std','Flow IAT Max',
'Flow IAT Min','Fwd IAT Total','Fwd IAT Mean','Fwd IAT Std','Fwd IAT Max','Fwd IAT Min','Bwd IAT Total',
'Bwd IAT Mean','Bwd IAT Std','Bwd IAT Max','Bwd IAT Min','Fwd PSH Flags','Bwd PSH Flags','Fwd URG Flags',
'Bwd URG Flags','Fwd Header Length','Bwd Header Length','Fwd Packets/s','Bwd Packets/s','Min Packet Length',
'Max Packet Length','Packet Length Mean','Packet Length Std','Packet Length Variance','FIN Flag Count',
'SYN Flag Count','RST Flag Count','PSH Flag Count','ACK Flag Count','URG Flag Count','CWE Flag Count',
'ECE Flag Count','Down/Up Ratio','Average Packet Size','Avg Fwd Segment Size','Avg Bwd Segment Size',
'Fwd Avg Bytes/Bulk','Fwd Avg Packets/Bulk','Fwd Avg Bulk Rate','Bwd Avg Bytes/Bulk','Bwd Avg Packets/Bulk',
'Bwd Avg Bulk Rate','Subflow Fwd Packets','Subflow Fwd Bytes','Subflow Bwd Packets','Subflow Bwd Bytes',
'Init_Win_bytes_forward','Init_Win_bytes_backward','act_data_pkt_fwd','min_seg_size_forward','Active Mean',
'Active Std','Active Max','Active Min','Idle Mean','Idle Std','Idle Max','Idle Min','protocol']
CTX_CIC = ['cve_ctx_service','cve_ctx_pool_size','cve_ctx_n_epss','cve_ctx_epss_coverage',
           'cve_ctx_service_max_epss','cve_ctx_n_generic_os_matches','cve_ctx_n_specific_product_matches']

try:
    from IPython.display import display, Pretty
    _h = display(Pretty("старт…"), display_id=True)
    def progress(msg): _h.update(Pretty(f"[{time.strftime('%H:%M:%S')}] {msg}"))
except Exception:
    def progress(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

REPORT, ERRORS = [], []
def R(line=""): REPORT.append(line)
def safe(n): return re.sub(r"[^0-9A-Za-z_]", "_", n)

def parse_port(v):
    s = str(v).strip().lower()
    try: return float(int(s, 16)) if s.startswith("0x") else float(s)
    except ValueError: return np.nan

# ---------------- завантаження і спліти ----------------
def load_unsw():
    df = pd.read_parquet(UNSW_PATH); df.columns = [c.strip() for c in df.columns]
    df["ct_ftp_cmd"] = pd.to_numeric(df["ct_ftp_cmd"].replace(" ", np.nan), errors="coerce")
    y = pd.to_numeric(df["Label"], errors="coerce").astype(int).to_numpy()
    t = pd.to_numeric(df["Stime"])
    b1, b2 = t.quantile(0.70, interpolation="nearest"), t.quantile(0.85, interpolation="nearest")
    split = np.where(t < b1, 0, np.where(t < b2, 1, 2))
    block = (t.to_numpy() // 60).astype(np.int64)
    raw = df["dsport"].astype(str)
    port = raw.map({u: parse_port(u) for u in raw.unique()}).astype(float).to_numpy()
    return dict(name="UNSW-NB15", split_desc="хронологічний спліт статті (Stime 0.70/0.85)", df=df, y=y,
                split=split, block=block, port=port, F0=F0_UNSW, CTX=CTX_UNSW, CAT={"state", "proto", "service"})

def load_cic():
    df = pd.read_parquet(CIC_PATH); df.columns = [c.strip() for c in df.columns]
    y = (df["attack_label"].astype(str) != "BENIGN").astype(int).to_numpy()
    raw = df["Timestamp"]
    ts = pd.to_datetime(raw) if np.issubdtype(raw.dtype, np.datetime64) else pd.to_datetime(raw.astype(str), dayfirst=True, errors="coerce")
    ts = ts.where(ts.dt.hour >= 8, ts + pd.Timedelta(hours=12))
    grp = ts.dt.date.astype(str) + "|" + df["attack_label"].astype(str)
    tn = ts.astype("int64")
    b1 = tn.groupby(grp).transform(lambda s: s.quantile(0.70, interpolation="nearest"))
    b2 = tn.groupby(grp).transform(lambda s: s.quantile(0.85, interpolation="nearest"))
    split = np.where(tn < b1, 0, np.where(tn < b2, 1, 2))
    block = pd.factorize(ts.dt.floor("min"))[0].astype(np.int64)
    port = pd.to_numeric(df["destination_port"], errors="coerce").astype(float).to_numpy()
    return dict(name="CIC-IDS2017", split_desc="виправлені таймстемпи, стратифікований часовий спліт (день×клас 70/15/15)",
                df=df, y=y, split=split, block=block, port=port, F0=F0_CIC, CTX=CTX_CIC, CAT={"protocol"})


# ---------------- Phase I / Phase II на виправлених сплітах ----------------
def entropy_bits(codes):
    _, cnt = np.unique(codes, return_counts=True)
    p = cnt / cnt.sum()
    return float(-(p * np.log2(p)).sum())

def joint_codes(*cols):
    code = None
    for c in cols:
        k = pd.factorize(pd.Series(c), use_na_sentinel=False)[0].astype(np.int64)
        code = k if code is None else pd.factorize(code * (int(k.max()) + 1) + k)[0].astype(np.int64)
    return code

def overlap(Ccodes, Fcodes):
    HC = entropy_bits(Ccodes); HF = entropy_bits(Fcodes)
    HCF = entropy_bits(joint_codes(Ccodes, Fcodes))
    HC_given_F = HCF - HF
    MI = HC - HC_given_F
    NMI = MI / min(HC, HF) if min(HC, HF) > 0 else float("nan")
    rho = HC_given_F / HC if HC > 0 else float("nan")
    return dict(H_F=HF, H_C=HC, H_C_given_F=max(HC_given_F, 0.0), MI=MI, NMI=NMI, rho=max(rho, 0.0))

def run(D):
    name, df, y, split, port = D["name"], D["df"], D["y"], D["split"], D["port"]
    F0, CTX, CAT = D["F0"], D["CTX"], D["CAT"]
    tr, te = split == 0, split == 2
    progress(f"{name}: Phase I")
    svc = df["cve_ctx_service"].astype(str).to_numpy()
    proto_col = "proto" if "proto" in df.columns else "protocol"
    proto = df[proto_col].astype(str).to_numpy()
    port_s = pd.Series(port).astype(str).to_numpy()
    ctx_all = joint_codes(*[df[c].to_numpy()[tr] for c in CTX])
    rows = []
    native_name = "service" if name.startswith("UNSW") else "protocol"
    native = df[native_name].astype(str).to_numpy()
    Csvc = joint_codes(svc[tr])
    rows.append(("cve_ctx_service", native_name, overlap(Csvc, joint_codes(native[tr]))))
    rows.append(("cve_ctx_service", f"({proto_col}, dst port)", overlap(Csvc, joint_codes(proto[tr], port_s[tr]))))
    rows.append(("усі CTI-поля разом", f"({proto_col}, dst port)", overlap(ctx_all, joint_codes(proto[tr], port_s[tr]))))

    progress(f"{name}: Phase II (F0 і F1, {ROUNDS} дерев)")
    X = df[F0 + CTX].reset_index(drop=True).copy()
    for c in set(CAT) | {"cve_ctx_service"}:
        X[c] = pd.Categorical(X[c], categories=pd.Categorical(X.loc[tr, c]).categories)
    spw = (y[tr] == 0).sum() / y[tr].sum()
    params = dict(objective="binary", metric="None", seed=SEED, num_leaves=15, learning_rate=0.05,
                  min_data_in_leaf=50, verbosity=-1, scale_pos_weight=spw)
    gains, aps = {}, {}
    for tag, cols in [("F0", F0), ("F1", F0 + CTX)]:
        cc = [c for c in cols if c in set(CAT) | {"cve_ctx_service"}]
        m = lgb.train(params, lgb.Dataset(X.loc[tr, cols], label=y[tr], categorical_feature=cc), num_boost_round=ROUNDS)
        g = pd.Series(m.feature_importance(importance_type="gain"), index=cols)
        gains[tag] = g / g.sum()
        aps[tag] = average_precision_score(y[te], m.predict(X.loc[te, cols]))
        del m; gc.collect()
    g1 = gains["F1"]
    other = [c for c in CTX if c != "cve_ctx_service"]
    top0 = gains["F0"].sort_values(ascending=False).head(5)

    R("=" * 96); R(f"{name}: {D['split_desc']}")
    R(f"контроль відтворення: AP(F0@{ROUNDS})={aps['F0']:.6f}, AP(F1@{ROUNDS})={aps['F1']:.6f}  "
      f"(очікувано з експерименту B: {'0.982439 / 0.983854' if name.startswith('UNSW') else '0.998869 / 0.999402'})")
    R(""); R("PHASE I (train partition):")
    R(f"  {'C':<20} {'F':<22} {'H(F)':>7} {'H(C)':>7} {'H(C|F)':>8} {'MI':>7} {'NMI':>7} {'rho':>7}")
    for c, f, o in rows:
        R(f"  {c:<20} {f:<22} {o['H_F']:>7.4f} {o['H_C']:>7.4f} {o['H_C_given_F']:>8.4f} {o['MI']:>7.4f} {o['NMI']:>7.4f} {o['rho']:>7.4f}")
    R(""); R("PHASE II (gain-share, F1 @ fixed budget):")
    R(f"  total CTI = {100 * g1[CTX].sum():.2f}%;  cve_ctx_service = {100 * g1['cve_ctx_service']:.2f}%;  інші CTI = {100 * g1[other].sum():.2f}%")
    R("  топ-5 ознак F0 (частка gain у F0 -> у F1):")
    for f, v in top0.items():
        R(f"    {f:<28} {100 * v:6.2f}% -> {100 * g1.get(f, 0):6.2f}%")
    R("  топ-5 ознак F1:")
    for f, v in g1.sort_values(ascending=False).head(5).items():
        R(f"    {f:<28} {100 * v:6.2f}%")
    del X; gc.collect()

t0 = time.time()
for loader in (load_unsw, load_cic):
    try:
        progress(f"завантаження: {loader.__name__}")
        D = loader(); run(D); del D; gc.collect()
    except Exception as e:
        import traceback
        ERRORS.append(f"{loader.__name__}: {type(e).__name__}: {e}\n{traceback.format_exc()}")
R(""); R(f"Час виконання: {(time.time() - t0) / 60:.1f} хв")
if ERRORS:
    R(""); R("ПОМИЛКИ:"); [R(e) for e in ERRORS]
report = "\n".join(REPORT)
with open(os.path.join(OUT, "phase12_report.txt"), "w") as fh:
    fh.write(report)
progress("готово — звіт нижче")
print(report)


[15:44:39] готово — звіт нижче

UNSW-NB15: хронологічний спліт статті (Stime 0.70/0.85)
контроль відтворення: AP(F0@500)=0.982439, AP(F1@500)=0.983854  (очікувано з експерименту B: 0.982439 / 0.983854)

PHASE I (train partition):
  C                    F                         H(F)    H(C)   H(C|F)      MI     NMI     rho
  cve_ctx_service      service                 1.9093  2.1454   0.6542  1.4911  0.7810  0.3050
  cve_ctx_service      (proto, dst port)       9.1445  2.1454   0.0000  2.1454  1.0000  0.0000
  усі CTI-поля разом   (proto, dst port)       9.1445  2.1454   0.0000  2.1454  1.0000  0.0000

PHASE II (gain-share, F1 @ fixed budget):
  total CTI = 0.24%;  cve_ctx_service = 0.21%;  інші CTI = 0.03%
  топ-5 ознак F0 (частка gain у F0 -> у F1):
    sttl                          96.24% ->  45.35%
    proto                          0.67% ->   0.34%
    ct_state_ttl                   0.65% ->   0.27%
    ct_srv_dst                     0.32% ->   0.47%
    service                        0.23% ->   0.01%
  топ-5 о